# HW01_4 — Image Features: Color Histogram + HOG

**AISV.X401 Deep Learning and Artificial Intelligence**  
**Student:** G.J. Johnson

Implements two global image feature extractors for RGB images:
- **(a)** Per-channel color histogram (16 bins/channel), concatenated → length 48 vector
- **(b)** HOG descriptor on the grayscale image (via scikit-image)

Given a list of N RGB images, `featurize_images` returns `X_feat` of shape `(N, F)`.

## 1. Imports

In [ ]:
import numpy as np
from skimage.feature import hog
from skimage.color import rgb2gray
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 2. Feature Extractor Implementations

In [ ]:
def color_histogram_rgb(img_rgb, bins=16):
    """
    Compute a normalized per-channel color histogram for an RGB image.

    Parameters
    ----------
    img_rgb : np.ndarray, shape (H, W, 3)
        RGB image with pixel values in [0, 1] (float) or [0, 255] (uint8).
    bins : int
        Number of histogram bins per channel (default 16).

    Returns
    -------
    hist_vec : np.ndarray, shape (bins * 3,)
        Concatenated, density-normalized histograms for R, G, B channels.
    """
    img_rgb = np.asarray(img_rgb, dtype=np.float64)

    # Normalize to [0, 255] range so bin edges are consistent regardless of
    # whether the caller passes float [0,1] or uint8 [0,255] images.
    if img_rgb.max() <= 1.0:
        img_rgb = img_rgb * 255.0

    channel_hists = []
    for c in range(3):  # R, G, B
        channel_pixels = img_rgb[:, :, c].ravel()
        hist, _ = np.histogram(channel_pixels, bins=bins, range=(0.0, 255.0),
                               density=True)
        channel_hists.append(hist)

    return np.concatenate(channel_hists)  # shape: (bins * 3,)

In [ ]:
def hog_feature(img_rgb):
    """
    Compute the HOG descriptor for an RGB image.

    The image is first converted to grayscale; scikit-image's `hog` is used
    for the actual descriptor computation.

    Parameters
    ----------
    img_rgb : np.ndarray, shape (H, W, 3)
        RGB image with pixel values in [0, 1] (float) or [0, 255] (uint8).

    Returns
    -------
    hog_vec : np.ndarray, shape (F_hog,)
        Flattened HOG feature vector.
    """
    img_rgb = np.asarray(img_rgb, dtype=np.float64)

    # rgb2gray expects values in [0, 1]
    if img_rgb.max() > 1.0:
        img_rgb = img_rgb / 255.0

    gray = rgb2gray(img_rgb)  # shape (H, W)

    hog_vec = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True,
    )

    return hog_vec

In [ ]:
def featurize_images(images_rgb, bins=16):
    """
    Extract concatenated color-histogram + HOG features for a list of images.

    Parameters
    ----------
    images_rgb : list of np.ndarray, each shape (H, W, 3)
        RGB images (float [0,1] or uint8 [0,255]).
    bins : int
        Bins per channel for the color histogram (default 16).

    Returns
    -------
    X_feat : np.ndarray, shape (N, F)
        Feature matrix; each row is [color_hist | hog_vec] for one image.
        F = bins*3 + F_hog.
    """
    feature_rows = []
    for img in images_rgb:
        color_hist = color_histogram_rgb(img, bins=bins)
        hog_vec    = hog_feature(img)
        combined   = np.concatenate([color_hist, hog_vec])
        feature_rows.append(combined)

    X_feat = np.vstack(feature_rows)  # shape (N, F)
    return X_feat

## 3. Sanity Checks

We generate three synthetic images and verify output shapes and basic properties.

In [ ]:
# --- Synthetic test images (64×64 RGB, float [0,1]) ---
rng = np.random.default_rng(seed=42)

img_random = rng.random((64, 64, 3))          # uniform random noise
img_red    = np.zeros((64, 64, 3));  img_red[:, :, 0] = 1.0   # solid red
img_green  = np.zeros((64, 64, 3));  img_green[:, :, 1] = 1.0 # solid green

images = [img_random, img_red, img_green]

# --- Individual function checks ---
BINS = 16
hist_vec = color_histogram_rgb(img_random, bins=BINS)
hog_vec  = hog_feature(img_random)

print(f"color_histogram_rgb output shape : {hist_vec.shape}  (expected ({BINS*3},) = (48,))")
print(f"hog_feature output shape         : {hog_vec.shape}")
print(f"color hist sum ≈ 1/bin × 3?      : {hist_vec.sum():.4f}  "
      f"(density=True → each channel sums to 1/bin_width; "
      f"all 3 channels ≈ {3*(1/(255/BINS)):.4f})")

# --- Full featurize check ---
X_feat = featurize_images(images, bins=BINS)
F_total = BINS * 3 + len(hog_vec)
print(f"\nfeaturize_images output shape    : {X_feat.shape}  "
      f"(expected (3, {F_total}))")
assert X_feat.shape == (3, F_total), "Shape mismatch!"
print("All shape assertions passed.")

## 4. Visualization

Plot the per-channel histograms for each synthetic image to verify the extractor
correctly captures the dominant colour.

In [ ]:
BINS = 16
labels  = ['Random noise', 'Solid red', 'Solid green']
colors  = [('red', 'green', 'blue')] * 3
ch_names = ['R', 'G', 'B']

fig, axes = plt.subplots(len(images), 1, figsize=(10, 8), tight_layout=True)

for ax, img, label in zip(axes, images, labels):
    hist_vec = color_histogram_rgb(img, bins=BINS)
    x = np.arange(BINS)
    width = 0.28
    for ci, (ch_label, col) in enumerate(zip(ch_names, ['red', 'limegreen', 'steelblue'])):
        ax.bar(x + ci * width, hist_vec[ci*BINS:(ci+1)*BINS],
               width=width, label=ch_label, color=col, alpha=0.75)
    ax.set_title(label)
    ax.set_xlabel('Bin index')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Per-channel Color Histograms (16 bins each)', fontsize=13)
plt.show()

## 5. HOG Visualization

Display the HOG image alongside the original to visually confirm gradient orientation is computed.

In [ ]:
from skimage.feature import hog as skimage_hog
from skimage.color import rgb2gray as skimage_rgb2gray

gray_random = skimage_rgb2gray(img_random)
_, hog_image = skimage_hog(
    gray_random,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
    feature_vector=True,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), tight_layout=True)
ax1.imshow(img_random)
ax1.set_title('Original (random noise)')
ax1.axis('off')
ax2.imshow(hog_image, cmap='gray')
ax2.set_title('HOG visualization')
ax2.axis('off')
plt.suptitle('HOG Feature Visualization', fontsize=13)
plt.show()

print(f"HOG feature vector length: {len(hog_feature(img_random))}")

## 6. Summary

| Function | Input | Output | Notes |
|---|---|---|---|
| `color_histogram_rgb` | `(H,W,3)` image | `(bins×3,)` vector | NumPy only; `density=True` |
| `hog_feature` | `(H,W,3)` image | `(F_hog,)` vector | scikit-image HOG on grayscale |
| `featurize_images` | list of N images | `(N, bins×3 + F_hog)` matrix | `np.vstack` of concatenated vectors |

Default settings: `bins=16` → 48-dim colour histogram; HOG with `orientations=9`, `pixels_per_cell=(8,8)`, `cells_per_block=(2,2)`, `block_norm='L2-Hys'`.